In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# Load The Data

In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


# Explore Patterns

In [4]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [5]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


In [6]:
class1 = train_data.loc[train_data.Pclass == 1]["Survived"]
rate_class1 = sum(class1)/len(class1)

print("% of class 1 who survived:", rate_class1)

% of class 1 who survived: 0.6296296296296297


In [7]:
class2 = train_data.loc[train_data.Pclass == 2]["Survived"]
rate_class2 = sum(class2)/len(class2)

print("% of class 2 who survived:", rate_class2)

% of class 2 who survived: 0.47282608695652173


In [8]:
class3 = train_data.loc[train_data.Pclass == 3]["Survived"]
rate_class3 = sum(class3)/len(class3)

print("% of class 3 who survived:", rate_class3)

% of class 3 who survived: 0.24236252545824846


In [9]:
family = train_data.loc[ (1+ train_data.SibSp + train_data.Parch) <5]["Survived"]
rate_family = sum(family)/len(family)

print("% of fare who survived:", rate_family)

% of fare who survived: 0.40048250904704463


# Data Preprocessing 

In [ ]:
features = ["Pclass", "Age", "Sex", "Fare", "Family", "SinglePassenger", "Embarked_C", "Embarked_Q", "Embarked_S",  "Children"]

# Decided to fill in age with a median value since the age varies
test_data["Age"] = test_data["Age"].fillna(test_data["Age"].median())
train_data["Age"] = train_data["Age"].fillna(train_data["Age"].median())

test_data["Sex"] = (test_data["Sex"] == "female").astype(int)
train_data["Sex"] = (train_data["Sex"] == "female").astype(int)

test_data["Fare"] = test_data["Fare"].fillna(test_data["Fare"].median())
train_data["Fare"] = train_data["Fare"].fillna(train_data["Fare"].median())

# I figured that most children under 18 would be put on lifeboats first so they would have a higher chance of surviving
test_data["Children"] = (test_data["Age"] < 18).astype(int)
train_data["Children"] = (train_data["Age"] < 18).astype(int)

# This includes the person himself and then his wife, children and siblings and all the family relations
test_data["Family"] = (1 + (test_data["Parch"]) + (test_data["SibSp"]))
train_data["Family"] = (1 + (train_data["Parch"]) + (train_data["SibSp"]))

test_data["SinglePassenger"] = (test_data["Family"] == 1).astype(int)
train_data["SinglePassenger"] = (train_data["Family"] == 1).astype(int)

test_data["Embarked"] = test_data["Embarked"].fillna(test_data["Embarked"].mode()[0])
train_data["Embarked"] = train_data["Embarked"].fillna(train_data["Embarked"].mode()[0])

# Variables for one hot encoding
test_cat = test_data[["Embarked"]]
train_cat = train_data[["Embarked"]]

# One Hot Encoding

In [11]:
from sklearn.preprocessing import OneHotEncoder

# I referenced the one hot encoding code from the end-to-end.ipynb on the median house prices like 
# Professor Chen suggested

#One Hot encoding for Embarked feature
cat_encoder = OneHotEncoder()

train_encoded = cat_encoder.fit_transform(train_cat)
test_encoded = cat_encoder.transform(test_cat)

train_encoded = train_encoded.toarray()
test_encoded = test_encoded.toarray()

embarked_cols = cat_encoder.get_feature_names_out(["Embarked"])
print(embarked_cols)

['Embarked_C' 'Embarked_Q' 'Embarked_S']


In [12]:

df_train = pd.DataFrame(train_encoded, columns = embarked_cols)
df_test = pd.DataFrame(test_encoded, columns = embarked_cols)

train_data = train_data.reset_index(drop = True)
test_data = test_data.reset_index(drop = True)

train_data = pd.concat([train_data, df_train], axis = 1)
test_data = pd.concat([test_data, df_test], axis = 1)

# since we add got new specific columns we need to drop the old generic Embarked column
train_data = train_data.drop("Embarked", axis=1)
test_data = test_data.drop("Embarked", axis=1)


# Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

X = train_data[features]
y = train_data["Survived"]
X_test = test_data[features]

model = RandomForestClassifier(n_estimators=240, max_depth=7, random_state=42)
model.fit(X,y)

#Performance Evaluation for Q3
training_prediction = model.predict(X)

accuracy = accuracy_score(y, training_prediction)
print("Accuracy:", accuracy)

cm = confusion_matrix(y, training_prediction)
print("Confusion Matrix:\n\n", cm)

f1 = f1_score(y, training_prediction)
print("F1 Score:", f1)


#Perform model.predict on the test data using the defined features for kaggle submission
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Accuracy: 0.8922558922558923
Confusion Matrix:

 [[532  17]
 [ 79 263]]
F1 Score: 0.8456591639871383
Your submission was successfully saved!
